In [ ]:
!git clone -q https://github.com/cad0m/jarida-rag.git

In [ ]:
!pip install -q "transformers>=4.51" sentence-transformers "psycopg[binary,pool]" pgvector mlflow pypdfium2 wordfreq bitsandbytes "ragas==0.2.15" "langchain-community<0.4" "langchain-core<1" "langchain-text-splitters<1" "langchain-huggingface<1"

In [ ]:
# Chunking lab: compares chunking variants on retrieval only (no LLM, no database), a few minutes in total
import os
import sys

from kaggle_secrets import UserSecretsClient

DAGSHUB_USER = "cad0m"
secrets = UserSecretsClient()

os.environ["JARIDA_PDF_DIR"] = "/kaggle/input/datasets/ayouberraouy/jarida"
os.environ["JARIDA_WORK"] = "/kaggle/working"
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/jarida-rag.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = secrets.get_secret("DAGSHUB_TOKEN")
sys.path.insert(0, "/kaggle/working/jarida-rag/src")

import mlflow
import pandas as pd
from jarida_rag import chunk, config, evaluate, store

mlflow.set_tracking_uri(config.MLFLOW_URI)
found = mlflow.search_experiments(filter_string="name = 'gazette-rag'")
experiment_id = found[0].experiment_id if found else mlflow.create_experiment("gazette-rag")
mlflow.set_experiment(experiment_id=experiment_id)

embedder = store.load_embedder()
reranker = store.load_reranker()

In [ ]:
# Each variant adds one change to the one before, so each row shows what that change does
variants = {
    "old":            {"CHUNK_HEADER": "first_200", "UNWRAP_LINES": False, "SPLIT_PREAMBLE": False},
    "old+unwrap":     {"CHUNK_HEADER": "first_200", "UNWRAP_LINES": True,  "SPLIT_PREAMBLE": False},
    "short-header":   {"CHUNK_HEADER": "short",     "UNWRAP_LINES": False, "SPLIT_PREAMBLE": False},
    "+unwrap-lines":  {"CHUNK_HEADER": "short",     "UNWRAP_LINES": True,  "SPLIT_PREAMBLE": False},
    "+split-preamble": {"CHUNK_HEADER": "short",    "UNWRAP_LINES": True,  "SPLIT_PREAMBLE": True},
}

results = {}
for name, settings in variants.items():
    for key, value in settings.items():
        setattr(config, key, value)
    chunks = chunk.chunk_all()
    vectors = store.embed(embedder, [c["embed_text"] for c in chunks])

    def search(question):
        return store.search_in_memory(chunks, vectors, embedder, question)

    def search_reranked(question):
        return store.search_in_memory(chunks, vectors, embedder, question, reranker)

    results[name] = evaluate.retrieval_eval(search, f"lab-v2-{name}", reranker_used=False)
    results[name + " +reranker"] = evaluate.retrieval_eval(search_reranked, f"lab-v2-{name}-reranker", reranker_used=True)
    results[name]["chunks"] = results[name + " +reranker"]["chunks"] = len(chunks)

In [ ]:
# Golden set v2 (86 questions). Golden set v1 gave evidence_mrr (plain / +reranker):
# old 0.951/0.977, old+unwrap 0.941/0.977, short-header 0.898/0.942, +unwrap 0.898/0.942, +split-preamble 0.888/0.922
# Keep the variant with the best evidence_mrr (the answer chunk ranked highest); hit_rate and mrr are near 1.0 for all
pd.DataFrame(results).T.round(3)[["chunks", "hit_rate", "mrr", "evidence_recall", "evidence_mrr", "avg_search_seconds"]]